In [ ]:
%load_ext autoreload
%autoreload 2

from enum import Enum
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
from IPython.display import display
from pathlib import Path
import utils
import numpy as np
import cv2
import triangle as tr
import matplotlib.pyplot as plt
import importlib

from dotenv import load_dotenv
load_dotenv()

client = genai.Client()

def query_image(contents: types.ContentListUnionDict):
    response = client.models.generate_content(
        model="gemini-2.5-flash-image-preview",
        contents=contents,
    )
    assert response.candidates is not None and len(response.candidates) > 0

    res_parts = response.candidates[0].content.parts
    assert isinstance(res_parts, list)
    img = None
    for part in res_parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            img = Image.open(BytesIO(part.inline_data.data))
            display(img)  # Display inline in notebook

    assert img is not None, "No image found in response"
    return img

output_folder = Path("./leaf_01")
output_folder.mkdir(parents=True, exist_ok=True)

class Map(Enum):
    COLOR = "color"
    BUMP = "bump"
    OCCLUSION = "occlusion"
    NORMAL = "normal"


def save_map(img: Image.Image, map_type: Map):
    img_path = output_folder / f"leaf_01_{map_type.value}.png"
    img.save(img_path)
    print(f"Saved {map_type.value} map to {img_path}")

def load_map(map_type: Map) -> Image.Image:
    img_path = output_folder / f"leaf_01_{map_type.value}.png"
    img = Image.open(img_path)
    print(f"Loaded {map_type.value} map from {img_path}")
    return img


In [ ]:

reference_image = Image.open("strawberry_leaf_reference.jpg")    
prompt = "\n".join([
    "Detailed photorealistic strawberry leaf color texture for a game engine",
    "The strawberry variety as shown in the reference image.",
    "No stems and the background is blue (far from any color in the leaf) for easy removal.",
    "Flat and have minimal shadows.(Like a texture map.)",
    "No highlights or shiny parts.",
    "No stems, only the leaf surface.",
    "Only the single terminal/central/apical leaflet of the trifoliate leaf.",
    "macro, close-up, high detail, high resolution",
    "ambient lighting, flat lighting, no shadows",
])

central_leaflet_color = query_image([reference_image, prompt])

In [ ]:


prompt = "\n".join([
    "Please refine this image texture of a strawberry leaf in the following ways:",
    "Remove the stem",
    # "remove shiny highlights",
    # "ambient lighting, flat lighting, no shadows",
])

central_leaflet_color_edited = query_image([central_leaflet_color, prompt])



In [ ]:
save_map(central_leaflet_color_edited, Map.COLOR)

In [ ]:
prompt = "\n".join([
    "This is the texture of the terminal/central/apical leaflet of the trifoliate strawberry leaf.",
    "Please create the bump map for this texture.",
    "A bump map is a grayscale image that represents surface height variations.",
    "The main stems are lowered in height compared to the leaf surface so they should be darker.",
])

central_leaflet_bump =  query_image([prompt, load_map(Map.COLOR)])


In [ ]:
save_map(central_leaflet_bump, Map.BUMP)

In [ ]:
from utils import triangulate_mask, create_alpha_mask, add_bones_to_mesh
from pxr import Usd

alpha_mask = create_alpha_mask(load_map(Map.COLOR), tolerance=50)
display(alpha_mask)

stage = Usd.Stage.CreateNew(str(output_folder / "leaf.usda"))
# triangulate_mask(alpha_mask, max_area=100000, epsilon=100.6, debug=True, stage=stage)
triangulate_mask(alpha_mask, max_area=1000, epsilon=2.6, debug=True, stage=stage)


root_bone_position = (0, 0.4)

midrib_bones = [(0.0, -0.2) for _ in range(4)]
secondary_bones = [
    [(70, -0.12), (-10, -0.12), (-10, -0.12)],
    [(70, -0.12), (-10, -0.12), (-10, -0.12)],
    [(70, -0.12), (-10, -0.12), (-10, -0.12)],
    [(70, -0.12), (-10, -0.12) ] 
]

add_bones_to_mesh(
    stage,
    mesh_path="/triangulated_mesh/mesh",
    root_position=root_bone_position,
    midrib_bones=midrib_bones,
    secondary_bones=secondary_bones,
    debug=True
)

# 

stage.Save()
del stage


In [ ]:
from pxr import Usd, UsdGeom, Kind, UsdSkel, Gf

def random_quat():
    u1, u2, u3 = np.random.rand(3)
    qx = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    qy = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    qz = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    qw = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return Gf.Quatf(qw, qx, qy, qz)

stage = Usd.Stage.CreateNew(str(output_folder / "leaves.usda"))
for i in range(500):
    anim = UsdSkel.Animation.Define(stage, f"/leaf_animation{i}")
    anim.CreateJointsAttr(["root_bone", "root_bone/mr0", "root_bone/mr0/mr1", "root_bone/mr0/mr1/mr2"])
    random_quaternion = np.random.rand(4)
    anim.CreateRotationsAttr([
        random_quat(),
        random_quat(),
        random_quat(),
        random_quat()
    ])
    anim.CreateTranslationsAttr([
        Gf.Vec3f(0, 0, 0),
        Gf.Vec3f(0, 0, 0),
        Gf.Vec3f(0, 0, 0),
        Gf.Vec3f(0, 0, 0)
    ])
    anim.CreateScalesAttr([
        Gf.Vec3f(1, 1, 1),
        Gf.Vec3f(1, 1, 1),
        Gf.Vec3f(1, 1, 1),
        Gf.Vec3f(1, 1, 1)
    ])

    leaf1 = stage.DefinePrim(f"/leaf{i}")
    leaf1.GetReferences().AddReference("./leaf.usda", "/Root")
    binding = UsdSkel.BindingAPI.Apply(leaf1)
    binding.CreateAnimationSourceRel().SetTargets([f"/leaf_animation{i}"])

    UsdGeom.Xformable(leaf1).AddTranslateOp(opSuffix='offset').Set(value=(0, i*10, 0))
    stage.Save()
del stage



In [ ]:


import numpy as np
vertices = np.array([(0.0, 0.0), (1.0, 0.0), (0.0, 1.0), (1.0, 1.0), (10.0, 10.0)])
joints = np.array([(0.0, 0.0), (0, 0), (1.0, 1.0), (0.5, 0.5)])
dist_matrix =   
print("Distance matrix:", dist_matrix)
# Find the two closest joints for each vertex
closest_joints = np.argsort(dist_matrix, axis=1)[:, :2]
print(f"Closest joints for each vertex: {closest_joints}")
distances = np.take_along_axis(dist_matrix, closest_joints, axis=1)
print(f"Distances to closest joints: {distances}")
# Normalize weights
# weights = 1.0 / (distances + 1e-8)
weights = distances / (np.sum(distances, axis=1, keepdims=True) + 1e-8)
print(f"Weights for closest joints: {weights}")
inversed_weights = 1.0 - weights
print(f"Inversed weights for closest joints: {inversed_weights}")


In [ ]:
def get_bounding_box_from_mask(mask):
    """
    Get the bounding box of non-black pixels from an alpha mask.
    
    Args:
        mask: PIL Image in grayscale mode ('L') where white=content, black=background
    
    Returns:
        tuple: (left, top, right, bottom) bounding box or None if no content found
    """
    # For grayscale masks, getbbox() finds non-zero pixels
    bbox = mask.getbbox()
    if bbox:
        print(f"Bounding box from mask: {bbox}")
        print(f"Size: {bbox[2] - bbox[0]} x {bbox[3] - bbox[1]}")
    else:
        print("No content found in mask")
    return bbox

def crop_images_to_bbox(images, bbox):
    """
    Crop a list of images to the same bounding box.
    
    Args:
        images: List of PIL Images
        bbox: tuple (left, top, right, bottom) bounding box
    
    Returns:
        List of cropped PIL Images, all with the same size
    """
    if not images or not bbox:
        return images
    
    cropped_images = []
    for img in images:
        cropped = img.crop(bbox)
        cropped_images.append(cropped)
    
    return cropped_images

# Get bounding box from the alpha mask
bbox = get_bounding_box_from_mask(alpha_mask)


# Now crop all images to the same bounding box
images_to_crop = [central_leaflet_color, central_leaflet_bump, alpha_mask]
cropped_images = crop_images_to_bbox(images_to_crop, bbox)

# Save and display the cropped images
central_leaflet_color_cropped = cropped_images[0]
central_leaflet_bump_cropped = cropped_images[1]
central_leaflet_mask_cropped = cropped_images[2]

display(central_leaflet_color_cropped)
display(central_leaflet_bump_cropped)
display(central_leaflet_mask_cropped)

In [ ]:
import numpy as np

def bump_to_normal(bump_image, strength=1.0):
    """
    Convert a grayscale bump map to a normal map.
    
    Args:
        bump_image: PIL Image in grayscale or RGB (will use luminance)
        strength: Multiplier for the normal strength (default 1.0)
    
    Returns:
        PIL Image: Normal map in RGB format
    """
    # Convert to grayscale array if needed
    if bump_image.mode == 'RGBA':
        # Use RGB channels, ignore alpha
        bump_array = np.array(bump_image.convert('RGB'))
        bump_gray = np.dot(bump_array[...,:3], [0.299, 0.587, 0.114])
    elif bump_image.mode == 'RGB':
        bump_array = np.array(bump_image)
        bump_gray = np.dot(bump_array, [0.299, 0.587, 0.114])
    else:
        bump_gray = np.array(bump_image.convert('L'))
    
    # Normalize to 0-1 range
    bump_gray = bump_gray.astype(np.float32) / 255.0
    
    # Calculate gradients (surface derivatives)
    grad_x = np.zeros_like(bump_gray)
    grad_y = np.zeros_like(bump_gray)
    
    # Calculate X gradient (horizontal)
    grad_x[:, 1:] = bump_gray[:, 1:] - bump_gray[:, :-1]
    grad_x[:, 0] = grad_x[:, 1]  # Duplicate edge
    
    # Calculate Y gradient (vertical) 
    grad_y[1:, :] = bump_gray[1:, :] - bump_gray[:-1, :]
    grad_y[0, :] = grad_y[1, :]  # Duplicate edge
    
    # Apply strength multiplier
    grad_x *= strength
    grad_y *= strength
    
    # Create normal vectors
    # Normal = (-dx, -dy, 1) normalized
    normal_x = -grad_x
    normal_y = -grad_y  
    normal_z = np.ones_like(grad_x)
    
    # Normalize the normal vectors
    length = np.sqrt(normal_x**2 + normal_y**2 + normal_z**2)
    normal_x /= length
    normal_y /= length  
    normal_z /= length
    
    # Convert from [-1,1] to [0,255] range
    # Normal map uses: R=X, G=Y, B=Z
    normal_r = ((normal_x + 1.0) * 0.5 * 255).astype(np.uint8)
    normal_g = ((normal_y + 1.0) * 0.5 * 255).astype(np.uint8)  
    normal_b = ((normal_z + 1.0) * 0.5 * 255).astype(np.uint8)
    
    # Stack into RGB image
    normal_rgb = np.stack([normal_r, normal_g, normal_b], axis=-1)
    
    # Convert back to PIL Image
    normal_map = Image.fromarray(normal_rgb, 'RGB')
    
    return normal_map

# Generate normal map from your bump map
central_leaflet_normal_cropped = bump_to_normal(central_leaflet_bump_cropped, strength=2.0)
display(central_leaflet_normal_cropped)

In [ ]:
central_leaflet_color_cropped.putalpha(central_leaflet_mask_cropped)

images = {
    "central_leaflet_color_cropped": central_leaflet_color_cropped,
    "central_leaflet_mask_cropped": central_leaflet_mask_cropped,
    "central_leaflet_bump_cropped": central_leaflet_bump_cropped,
    "central_leaflet_normal_cropped": central_leaflet_normal_cropped,
}

for name, img in images.items():
    img = img.transpose(method=Image.FLIP_TOP_BOTTOM)
    # Resize to 1024x1024 for consistency
    img = img.resize((1024, 1024), resample=Image.LANCZOS)
    img.save(f"{name}.png")
    display(img)